# 1D-CNN IDS Model — ISOT Drone Dataset
**Source:** University of Victoria ISOT Lab  
**Attack Types:** Benign, DoS, Injection, Ip Spoofing, Manipulation, MITM, Password Cracking, Replay, Unauth, Video  
**Features:** 61 network traffic features  
**Output:** cnn_model.keras, cnn_results.json

In [1]:
# ============================================================
# cell 2 Loading data:
# WHAT: Load preprocessed dataset files for 1D-CNN model
#
# WHY:  CNN needs clean, scaled, numeric data to train.
#       Preprocessing already done — we just load results.
#       label_classes = converts numbers back to attack names
#       feature_names = needed later for SHAP and LIME
#       num_classes = tells CNN how many output neurons to use
#                     UAV_Attack=3, ISOT=10, UAVCAN=2
#
# HOW:  Step 1: np.load() → loads .npy arrays (features)
#       Step 2: pd.read_csv() → loads labels and names
#       Step 3: squeeze() → converts DataFrame to 1D array
#       Step 4: len(label_classes) → auto-detect num_classes
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import time

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISOT/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()
label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

num_classes = len(label_classes)

print(f"X_train: {X_train.shape}")

2026-05-16 11:18:28.776207: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


X_train: (205671, 61)


In [2]:
# ============================================================
# Cell 3 Reshape:
# WHAT: Reshape features from 2D to 3D for CNN input
#
# WHY:  CNN requires 3D input: (samples, timesteps, channels)
#       Our data is 2D: (samples, features)
#       We treat each feature as one timestep with 1 channel
#       Without reshape, CNN will crash with shape error
#
# HOW:  Before: (7046, 80)   ← 2D, samples × features
#       After:  (7046, 80, 1) ← 3D, samples × features × 1
#       X.shape[0] = number of samples (rows)
#       X.shape[1] = number of features (columns)
#       The final 1 = one channel (like grayscale image)
# ============================================================

X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train reshaped: {X_train_cnn.shape}")
print(f"X_test reshaped:  {X_test_cnn.shape}")
print("Reshape complete!")

X_train reshaped: (205671, 61, 1)
X_test reshaped:  (88145, 61, 1)
Reshape complete!


In [3]:
# ============================================================
# cell 4 Build CNN:
# WHAT: Build a 1D-CNN model architecture for IDS
#
# WHY:  1D-CNN treats features as a sequence and learns
#       local patterns between neighboring features.
#       Better than RF for detecting subtle sequential
#       patterns in network/sensor traffic data.
#       num_classes changes per dataset:
#       UAV_Attack=3, ISOT=10, UAVCAN=2
#
# HOW:  Layer 1: Conv1D(64) → learn local feature patterns
#                kernel_size=3 → looks at 3 features at once
#                relu → keeps positive values, sets negative=0
#       Layer 2: MaxPooling → reduce size by half (80→40)
#       Layer 3: Conv1D(128) → learn deeper patterns
#       Layer 4: MaxPooling → reduce again (40→20)
#       Layer 5: Flatten → convert 2D to 1D vector
#       Layer 6: Dense(128) → fully connected layer
#       Layer 7: Dropout(0.3) → randomly drop 30% neurons
#                to prevent overfitting (memorizing training data)
#       Layer 8: Dense(num_classes, softmax) → output layer
#                softmax = converts scores to probabilities
#                one probability per attack class
# ============================================================

print("Building 1D-CNN model...")

model = keras.Sequential([
    keras.layers.Input(shape=(61, 1)),
    keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',               # adaptive learning rate optimizer
    loss='sparse_categorical_crossentropy',  # for integer labels (0,1,2...)
    metrics=['accuracy']
)

model.summary()
print("CNN model built!")

Building 1D-CNN model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 61, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 30, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       245,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 272,138 (1.04 MB)

 Trainable params: 272,138 (1.04 MB)

 Non-trainable params: 0 (0.00 B)

CNN model built!


In [4]:
# ============================================================
# cell 5 Train:
# WHAT: Train the 1D-CNN on training data
#
# WHY:  fit() = model learns from X_train and y_train
#       epochs=5 = 5 full passes through training data
#       batch_size=256 = process 256 samples at a time
#       validation_split=0.1 = use 10% of train data
#       to monitor overfitting during training
#       Training time is recorded as an SE metric
#
# HOW:  Step 1: record start time
#       Step 2: model.fit() trains the CNN
#       Step 3: record end time
#       Step 4: print training summary
# ============================================================

print("Training 1D-CNN...")
start_time = time.time()

history = model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

cnn_train_time = round(time.time() - start_time, 2)
print(f"Training complete! Time: {cnn_train_time}s")

Training 1D-CNN...
Epoch 1/5
724/724 ━━━━━━━━━━━━━━━━━━━━ 52s 68ms/step - accuracy: 0.9549 - loss: 0.1993 - val_accuracy: 0.9859 - val_loss: 0.0374
Epoch 2/5
724/724 ━━━━━━━━━━━━━━━━━━━━ 78s 63ms/step - accuracy: 0.9889 - loss: 0.0334 - val_accuracy: 0.9947 - val_loss: 0.0204
Epoch 3/5
724/724 ━━━━━━━━━━━━━━━━━━━━ 57s 79ms/step - accuracy: 0.9928 - loss: 0.0195 - val_accuracy: 0.9952 - val_loss: 0.0157
Epoch 4/5
724/724 ━━━━━━━━━━━━━━━━━━━━ 49s 67ms/step - accuracy: 0.9941 - loss: 0.0161 - val_accuracy: 0.9947 - val_loss: 0.0116
Epoch 5/5
724/724 ━━━━━━━━━━━━━━━━━━━━ 47s 65ms/step - accuracy: 0.9958 - loss: 0.0116 - val_accuracy: 0.9953 - val_loss: 0.0111
Training complete! Time: 284.0s


In [5]:
# ============================================================
# cell 6 Evaluate:
# WHAT: Evaluate CNN on unseen test data
#
# WHY:  model.predict() returns probability for each class
#       np.argmax() picks the highest probability class
#       classification_report shows F1 per attack type
#       target_names converts numbers back to attack names
#       weighted average accounts for class imbalance
#
# HOW:  Step 1: predict probabilities for all test samples
#       Step 2: argmax → pick highest probability class
#       Step 3: calculate Accuracy, Precision, Recall, F1
#       Step 4: print detailed per-class report
# ============================================================

print("Evaluating CNN model...")
start_time = time.time()

y_pred_proba = model.predict(X_test_cnn, batch_size=256, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

cnn_pred_time = round(time.time() - start_time, 2)

acc = accuracy_score(y_test, y_pred)
p   = precision_score(y_test, y_pred, average='weighted')
r   = recall_score(y_test, y_pred, average='weighted')
f1  = f1_score(y_test, y_pred, average='weighted')

print(f"\n=== 1D-CNN Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Training time:   {cnn_train_time}s")
print(f"Prediction time: {cnn_pred_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=label_classes))

Evaluating CNN model...
345/345 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step

=== 1D-CNN Results ===
Accuracy:  0.9961 (99.61%)
Precision: 0.9962 (99.62%)
Recall:    0.9961 (99.61%)
F1 Score:  0.9960 (99.60%)
Training time:   284.0s
Prediction time: 5.68s

Detailed Report:
                   precision    recall  f1-score   support

           Benign       1.00      1.00      1.00     38421
              DoS       1.00      1.00      1.00     36373
        Injection       0.99      0.99      0.99       142
      Ip_Spoofing       0.99      1.00      0.99       292
             MITM       0.95      1.00      0.97      3154
     Manipulation       0.60      0.42      0.49       183
Password_Cracking       1.00      1.00      1.00      6937
           Replay       0.54      0.72      0.61       176
           Unauth       1.00      1.00      1.00       150
            Video       1.00      0.92      0.96      2317

         accuracy                           1.00     88145
        macro avg       0.9

In [6]:
# Cell 7 Save:
import json
import os

model.save(save_path + "cnn_model.keras")

cnn_results = {
    "model": "1D-CNN",
    "dataset": "ISOT",
    "accuracy": round(acc, 4),
    "precision": round(p, 4),
    "recall": round(r, 4),
    "f1_score": round(f1, 4),
    "training_time_seconds": cnn_train_time,
    "prediction_time_seconds": cnn_pred_time,
    "epochs": 5,
    "batch_size": 256
}

with open(save_path + "cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=4)

print("Model saved: cnn_model.keras")
print("Results saved: cnn_results.json")

Model saved: cnn_model.keras
Results saved: cnn_results.json


## CNN Results Summary — ISOT Drone Dataset

| Metric | Value |
|--------|-------|
| Dataset | ISOT Drone Dataset (University of Victoria) |
| Model | 1D-CNN (5 epochs, batch=256) |
| Train set | 205,671 rows |
| Test set | 88,145 rows |
| Features | 61 network traffic features |
| Accuracy | 99.61% |
| Precision | 99.62% |
| Recall | 99.61% |
| F1 Score | 99.60% |
| Training time | 284.0s |
| Prediction time | 5.68s |

### Per-Class Results
| Class | F1 | Samples |
|-------|----|---------|
| Benign | 100% | 38,421 |
| DoS | 100% | 36,373 |
| Password_Cracking | 100% | 6,937 |
| Video | 96% | 2,317 |
| MITM | 97% | 3,154 |
| Injection | 99% | 142 |
| Ip_Spoofing | 99% | 292 |
| Unauth | 100% | 150 |
| Replay | 61% | 176 |
| Manipulation | 49% | 183 |

### Key Finding
CNN achieved **99.60% F1** overall. However class imbalance
causes low F1 for minority classes:
- Manipulation F1 = 49% (only 183 test samples)
- Replay F1 = 61% (only 176 test samples)
This confirms class imbalance significantly impacts CNN performance
on rare attack types — a key benchmark finding.